# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² Mental Health Survey dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL. This dataset contains survey data on mental health indicators among residents of Kilifi County, including demographics and scores from GAD-7, PHQ-9, and PSQ assessments.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. The Croissant schema URL is provided below.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their IDs. All entities will be referenced by their `@id` fields for consistency.

In [ ]:
# List available record sets and their fields by @id
record_sets = list(dataset.record_sets)
print(f"Number of record sets: {len(record_sets)}")
for record_set in record_sets:
    print(f"RecordSet @id: {record_set['@id']}, Name: {record_set.get('name', 'N/A')}")
    fields = record_set.get('field', [])
    # Special handling for single field entries
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        print(f"  Field @id: {field['@id']}, Name: {field.get('name', 'N/A')}")
    columns = record_set.get('column', [])
    if isinstance(columns, dict):
        columns = [columns]
    for column in columns:
        print(f"  Column @id: {column['@id']}, Name: {column.get('name', 'N/A')}")


### Preview Records from a Record Set
Print the first few records by their field/column `@id` for review.

In [ ]:
# Choose the major survey record set
# Here, we select the first RecordSet for demonstration
if record_sets:
    main_record_set_id = record_sets[0]['@id']
    print(f"Previewing records from RecordSet: {main_record_set_id}")
    for i, x in enumerate(dataset.records(record_set=main_record_set_id)):
        if i < 3:
            print(x)
        else:
            break


## 3. Data Extraction
Load data from available record sets into DataFrames for analysis. Use the record set and field/column `@id`s identified in the overview.

In [ ]:
# Extract data from each record set
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print("\nRecordSet @id: {} Columns: {}".format(rs_id, df.columns.tolist()))

# Show head for the main record set
if record_set_ids:
    main_df = dataframes[record_set_ids[0]]
    print(main_df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by key attributes. Use only `@id` to reference fields/columns.

In [ ]:
import numpy as np
# Example: Analyze numeric field (GAD-7 total score) in the main record set
# Find a numeric field @id, e.g., GAD-7 score. Let's assume it is 'gad7_total_score', adjust as needed from columns printed above.
numeric_field_id = None
for col in main_df.columns:
    if 'gad7' in col.lower() and ('score' in col.lower() or 'total' in col.lower()):
        numeric_field_id = col
        break
# Fallback, try 'phq9_total_score' or 'psq_total_score'
if numeric_field_id is None:
    for col in main_df.columns:
        if 'phq9' in col.lower() and 'score' in col.lower():
            numeric_field_id = col
            break
if numeric_field_id is None:
    for col in main_df.columns:
        if 'psq' in col.lower() and 'score' in col.lower():
            numeric_field_id = col
            break

if numeric_field_id:
    print(f"Using numeric field for EDA: {numeric_field_id}")
    threshold = main_df[numeric_field_id].mean()  # Use mean as threshold
    filtered_df = main_df[main_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())
    # Normalize the scores
    col_normed = f"{numeric_field_id}_normalized"
    filtered_df[col_normed] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, col_normed]].head())

    # Group by a key demographic column
    group_field_id = None
    for col in main_df.columns:
        if 'gender' in col.lower():
            group_field_id = col
            break
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No appropriate numeric field found in main record set.")

## 5. Visualization
Visualize data distributions and relationships between fields, referencing them by their `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize distribution of GAD-7/PHQ-9/PSQ scores by a demographic field
if numeric_field_id and group_field_id:
    plt.figure(figsize=(8, 5))
    sns.boxplot(data=main_df, x=group_field_id, y=numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id} by {group_field_id}")
    plt.ylabel(numeric_field_id)
    plt.xlabel(group_field_id)
    plt.tight_layout()
    plt.show()

# Plot histogram of the numeric score
if numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(main_df[numeric_field_id], kde=True, bins=20)
    plt.title(f"Histogram of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()

## 6. Conclusion
This notebook demonstrated guided exploration of the FAIR² Mental Health Survey dataset using `mlcroissant`. Key steps included referencing data entities by their `@id`, loading DataFrames, and performing basic analysis and visualization. The dataset provides insight into mental health indicators and demographic distributions in Kilifi County, Kenya—serving as a valuable resource for AI-ready public health research.